In [174]:
import re
import csv
from datetime import date
from typing import Dict, List

# Departamentos válidos para empleados
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']

# Series válidas para facturas
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [175]:
#Funciones extra
def sugerir_correccion(codigo: str) -> str:
    """Sugiere corrección convirtiendo a mayúsculas."""
    return codigo.upper()

def validar_fecha_real(anio: int, mes: int, dia: int) -> bool:
    """Valida que una fecha sea real en el calendario."""
    try:
        date(anio, mes, dia)
        return True
    except ValueError:
        return False

def exportar_resultados(reporte: Dict, archivo: str) -> None:
    """Exporta el reporte a un archivo CSV."""
    with open(archivo, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Codigo', 'Tipo', 'Valido', 'Detalles'])
        for item in reporte['detalle']:
            detalles_str = str(item['detalles']) if item['valido'] else ''
            writer.writerow([item['codigo'], item['tipo'], item['valido'], detalles_str])

In [176]:
#Validadores individuales (validar producto).

def validar_producto(codigo: str) -> Dict:
    """
    Valida código de producto.
    Formato: ABC-1234-MX (3 letras - 4 dígitos - 2 letras país)
    """
    resultado = {"valido": False, "categoria": None, "numero": None, "pais": None}

    patron = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'
    match = re.match(patron, codigo)

    if match:
        resultado.update({
            "valido": True,
            "categoria": match.group(1),
            "numero": match.group(2),
            "pais": match.group(3)
        })
    return resultado

In [177]:
#Validadores individuales (validar envio).

def validar_envio(codigo: str) -> Dict:
    """
    Valida código de envío.
    Formato: ENV-YYYY-MM-DD-NNNNNN
    """
    resultado = {"valido": False, "fecha": None, "secuencial": None}

    patron = r'^ENV-(202[0-9]|2030)-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        anio, mes, dia, seq = match.groups()
        # Se usa la función Bonus para validar la fecha
        if validar_fecha_real(int(anio), int(mes), int(dia)):
            resultado.update({
                "valido": True,
                "fecha": f"{anio}-{mes}-{dia}",
                "secuencial": seq
            })
    return resultado

In [178]:
#Validadores individuales (validar empleado).

def validar_empleado(codigo: str) -> Dict:
    """
    Valida código de empleado.
    Formato: EMP-XXX-NNNN
    """
    resultado = {"valido": False, "departamento": None, "numero": None}

    patron = r'^EMP-([A-Z]{3})-([1-9]\d{3})$'
    match = re.match(patron, codigo)

    if match:
        depto = match.group(1)
        if depto in DEPARTAMENTOS_VALIDOS:
            resultado.update({
                "valido": True,
                "departamento": depto,
                "numero": match.group(2)
            })
    return resultado

In [179]:
#Validadores individuales (validar factura).

def validar_factura(codigo: str) -> Dict:
    """
    Valida código de factura.
    Formato: FAC-S-NNNNNN
    """
    resultado = {"valido": False, "serie": None, "numero": None}

    patron = r'^FAC-([A-E])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        resultado.update({
            "valido": True,
            "serie": match.group(1),
            "numero": match.group(2)
        })
    return resultado

In [180]:
#Validador Universal.
def validar_codigo(codigo: str) -> Dict:
    resultado = {
        "codigo": codigo, "tipo": "desconocido",
        "valido": False, "detalles": {}
    }

    if codigo.upper().startswith("ENV-"):
        resultado["tipo"] = "envio"
        validacion = validar_envio(codigo)
    elif codigo.upper().startswith("EMP-"):
        resultado["tipo"] = "empleado"
        validacion = validar_empleado(codigo)
    elif codigo.upper().startswith("FAC-"):
        resultado["tipo"] = "factura"
        validacion = validar_factura(codigo)
    # Identifica la estructura básica de producto para agruparlo correctamente aunque sea inválido
    elif re.match(r'^[A-Za-z]+-\d+-[A-Za-z]+$', codigo):
        resultado["tipo"] = "producto"
        validacion = validar_producto(codigo)
    else:
        return resultado

    resultado["valido"] = validacion.pop("valido")
    if resultado["valido"]:
        resultado["detalles"] = validacion

    return resultado

In [181]:
#Procesamiento por lotes.
def procesar_lote(codigos: List[str]) -> Dict:
    resultado = {
        "total": len(codigos), "validos": 0, "invalidos": 0,
        "por_tipo": {
            "producto": {"total": 0, "validos": 0},
            "envio": {"total": 0, "validos": 0},
            "empleado": {"total": 0, "validos": 0},
            "factura": {"total": 0, "validos": 0},
            "desconocido": {"total": 0, "validos": 0}
        },
        "detalle": []
    }

    for codigo in codigos:
        res = validar_codigo(codigo)
        resultado["detalle"].append(res)

        tipo = res["tipo"]
        resultado["por_tipo"][tipo]["total"] += 1

        if res["valido"]:
            resultado["validos"] += 1
            resultado["por_tipo"][tipo]["validos"] += 1
        else:
            resultado["invalidos"] += 1

    return resultado

In [182]:
#Mostrar resultados.
def mostrar_resultado(resultado: Dict) -> None:
    estado = "✓" if resultado["valido"] else "✗"
    print(f"{estado} {resultado['codigo']:<30} | Tipo: {resultado['tipo']:<12}")
    if resultado["valido"] and resultado["detalles"]:
        detalles = ", ".join(f"{k}: {v}" for k, v in resultado["detalles"].items() if v)
        print(f"   └── {detalles}")

def mostrar_reporte(reporte: Dict) -> None:
    print("=" * 60)
    print("                 REPORTE DE VALIDACIÓN")
    print("=" * 60)
    print(f"\nTotal procesados: {reporte['total']}")
    if reporte['total'] > 0:
        print(f"Válidos: {reporte['validos']} ({reporte['validos']/reporte['total']*100:.1f}%)")
        print(f"Inválidos: {reporte['invalidos']} ({reporte['invalidos']/reporte['total']*100:.1f}%)")

    print("\nDesglose por tipo:")
    print("-" * 40)
    for tipo, stats in reporte["por_tipo"].items():
        if stats["total"] > 0:
            tasa = stats["validos"] / stats["total"] * 100
            print(f"  {tipo.capitalize():<12}: {stats['validos']:>3}/{stats['total']:<3} ({tasa:.0f}% válidos)")

    print("\n" + "=" * 60)

In [183]:
# Códigos de prueba
CODIGOS_PRUEBA = [
    # Productos
    "TEC-0001-MX",      # Válido
    "ALI-9999-US",      # Válido
    "ROB-1234-CA",      # Válido
    "tec-0001-MX",      # Inválido: minúsculas
    "TEC-001-MX",       # Inválido: solo 3 dígitos
    "TECH-0001-MX",     # Inválido: 4 letras en categoría

    # Envíos
    "ENV-2024-03-15-001234",    # Válido
    "ENV-2025-12-01-999999",    # Válido
    "ENV-2019-03-15-001234",    # Inválido: año fuera de rango
    "ENV-2024-13-15-001234",    # Inválido: mes 13
    "ENV-2024-03-32-001234",    # Inválido: día 32

    # Empleados
    "EMP-VEN-1234",     # Válido
    "EMP-TEC-9999",     # Válido
    "EMP-ADM-1000",     # Válido
    "EMP-VEN-0123",     # Inválido: empieza con 0
    "EMP-XXX-1234",     # Inválido: departamento no válido
    "EMP-VEN-123",      # Inválido: solo 3 dígitos

    # Facturas
    "FAC-A-123456",     # Válido
    "FAC-E-000001",     # Válido
    "FAC-B-999999",     # Válido
    "FAC-F-123456",     # Inválido: serie F no existe
    "FAC-A-12345",      # Inválido: solo 5 dígitos
    "FAC-a-123456",     # Inválido: serie en minúscula

    # Desconocidos
    "XXX-1234",         # Desconocido
    "RANDOM-CODE",      # Desconocido
]

In [184]:
#Prueba de validación individual
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)
print("\n-- Productos --")
print(validar_producto("TEC-0001-MX"))
print(validar_producto("tec-0001-MX"))

print("\n-- Envíos --")
print(validar_envio("ENV-2024-03-15-001234"))
print(validar_envio("ENV-2024-13-15-001234"))

print("\n-- Empleados --")
print(validar_empleado("EMP-VEN-1234"))
print(validar_empleado("EMP-VEN-0123"))

print("\n-- Facturas --")
print(validar_factura("FAC-A-123456"))
print(validar_factura("FAC-F-123456"))

print("\n\n")

#Prueba del validador universal
print("PRUEBA DE VALIDADOR UNIVERSAL")
print("=" * 50)
for codigo in CODIGOS_PRUEBA[:10]:
    mostrar_resultado(validar_codigo(codigo))

print("\n\n")

#Prueba de procesamiento por lotes
print("PRUEBA DE PROCESAMIENTO POR LOTES")
reporte = procesar_lote(CODIGOS_PRUEBA)
mostrar_reporte(reporte)

PRUEBA DE FUNCIONES INDIVIDUALES

-- Productos --
{'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}

-- Envíos --
{'valido': True, 'fecha': '2024-03-15', 'secuencial': '001234'}
{'valido': False, 'fecha': None, 'secuencial': None}

-- Empleados --
{'valido': True, 'departamento': 'VEN', 'numero': '1234'}
{'valido': False, 'departamento': None, 'numero': None}

-- Facturas --
{'valido': True, 'serie': 'A', 'numero': '123456'}
{'valido': False, 'serie': None, 'numero': None}



PRUEBA DE VALIDADOR UNIVERSAL
✓ TEC-0001-MX                    | Tipo: producto    
   └── categoria: TEC, numero: 0001, pais: MX
✓ ALI-9999-US                    | Tipo: producto    
   └── categoria: ALI, numero: 9999, pais: US
✓ ROB-1234-CA                    | Tipo: producto    
   └── categoria: ROB, numero: 1234, pais: CA
✗ tec-0001-MX                    | Tipo: producto    
✗ TEC-001-MX                     | Tipo: producto

##Explicación de los Patrones Regex Utilizados

### 1. Código de Producto
**Patrón:** `^([A-Z]{3})-(\d{4})-([A-Z]{2})$`
* `^` y `$`: Anclajes para asegurar que la cadena completa coincida con el patrón de inicio a fin.
* `([A-Z]{3})`: **Grupo 1 (Categoría).** Extrae exactamente 3 letras mayúsculas de la A a la Z.
* `-`: Coincide con el guion separador literal.
* `(\d{4})`: **Grupo 2 (Número).** Extrae exactamente 4 dígitos numéricos.
* `([A-Z]{2})`: **Grupo 3 (País).** Extrae exactamente 2 letras mayúsculas al final.

### 2. Código de Envío
**Patrón:** `^ENV-(202[0-9]|2030)-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])-(\d{6})$`
* `^ENV-`: Asegura que el código inicie estrictamente con el prefijo "ENV-".
* `(202[0-9]|2030)`: **Grupo 1 (Año).** Valida y extrae el año limitando el rango desde 2020 hasta 2029 (`202[0-9]`) o exactamente `2030`.
* `(0[1-9]|1[0-2])`: **Grupo 2 (Mes).** Valida y extrae el mes, permitiendo de `01` a `09`, o `10`, `11`, `12`.
* `(0[1-9]|[12]\d|3[01])`: **Grupo 3 (Día).** Valida y extrae el día, permitiendo `01` a `09`, `10` a `29` (`[12]\d`), o `30` y `31`.
* `(\d{6})`: **Grupo 4 (Secuencial).** Extrae exactamente 6 dígitos finales.

### 3. Código de Empleado
**Patrón:** `^EMP-([A-Z]{3})-([1-9]\d{3})$`
* `^EMP-`: Asegura el prefijo obligatorio "EMP-".
* `([A-Z]{3})`: **Grupo 1 (Departamento).** Extrae 3 letras mayúsculas que representan el departamento (la validación de si pertenece a la lista permitida se hace por código).
* `([1-9]\d{3})`: **Grupo 2 (Número).** Exige que el primer dígito sea del 1 al 9 (para evitar que inicie con 0) seguido de exactamente 3 dígitos numéricos cualquiera.

### 4. Código de Factura
**Patrón:** `^FAC-([A-E])-(\d{6})$`
* `^FAC-`: Asegura el prefijo obligatorio "FAC-".
* `([A-E])`: **Grupo 1 (Serie).** Permite y extrae un solo carácter que debe ser exclusivamente A, B, C, D o E.
* `(\d{6})`: **Grupo 2 (Número).** Extrae exactamente 6 dígitos numéricos correspondientes al folio.